This notebook integrates gpt4o for question answering on information in the KG.

In [1]:
from neo4j import GraphDatabase
from pathlib import Path
import os
from dotenv import load_dotenv
import openai
import tkinter as tk
from tkinter import scrolledtext

In [2]:
dotenv_path = Path('~/.env').expanduser()
load_dotenv(dotenv_path=dotenv_path)  

# Get variables
POKEDEX_URI = os.getenv('POKEDEX_URI')
POKEDEX_USER = os.getenv('POKEDEX_USER')
POKEDEX_PASSWORD = os.getenv('POKEDEX_PASSWORD')
#database_name = os.getenv('DATABASE_NAME')

driver = GraphDatabase.driver(POKEDEX_URI, auth=(POKEDEX_USER, POKEDEX_PASSWORD))
driver.verify_connectivity()

In [3]:
def get_graph_schema():
    schema = {}
    with driver.session() as session:
        # Node properties
        node_props = session.run("""
            CALL db.schema.nodeTypeProperties()
            YIELD nodeLabels, propertyName, propertyTypes
            RETURN nodeLabels, propertyName, propertyTypes
        """).data()
        
        # Relationship properties
        rel_props = session.run("""
            CALL db.schema.relTypeProperties()
            YIELD relType, propertyName, propertyTypes
            RETURN relType, propertyName, propertyTypes
        """).data()
        
        schema["nodes"] = node_props
        schema["relationships"] = rel_props
    return schema

client = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def ask_gpt4o(prompt):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content

def run_cypher(query):
    with driver.session() as session:
        return session.run(query).data()

def question_to_cypher_and_answer(question):
    # prompt LLM to translate the question into a Cypher query

    schema = get_graph_schema()
    
    prompt = f"""
    You are a Cypher expert. Here is the schema of the graph database: 
    {schema}
    Given the following question, respond ONLY with the Cypher query. Take into account the schema you have to figure out what the node, relationship and property names are.
    Do NOT include markdown or code block formatting. Just return the plain Cypher.

    Question: "{question}"
    """
    cypher_query = ask_gpt4o(prompt)
    print("Generated Cypher Query:\n", cypher_query)

    results = run_cypher(cypher_query)
    print("Query Results:\n", results)

    interpretation_prompt = f"""
    Given the question: "{question}"
    And the results: {results}
    List the results. Do not interpret them and do not check whether their are factually correct.
    """
    explanation = ask_gpt4o(interpretation_prompt)
    return explanation

In [ ]:
question = "What type is Empoleon?"
print(question_to_cypher_and_answer(question))

--------------------------------------------------------------------------------------------------------- **Visualisation!** ---------------------------------------------------------------------------------------------------------

In [8]:
def question_to_cypher_and_answer(question):
    schema = get_graph_schema()
    
    prompt = f"""
    You are a Cypher expert. Here is the schema of the graph database: 
    {schema}
    Given the following question, respond ONLY with the Cypher query. Take into account the schema you have to figure out what the node, relationship and property names are.
    Do NOT include markdown or code block formatting. Just return the plain Cypher.

    Question: "{question}"
    """
    cypher_query = ask_gpt4o(prompt)
    print("Generated Cypher Query:\n", cypher_query)

    results = run_cypher(cypher_query)
    print("Query Results:\n", results)

    interpretation_prompt = f"""
    Given the question: "{question}"
    And the results: {results}
    List the results. Do not interpret them and do not check whether their are factually correct.
    """
    explanation = ask_gpt4o(interpretation_prompt)
    return explanation

def ask_question():
    user_input = entry.get()
    if not user_input.strip():
        return
    response = question_to_cypher_and_answer(user_input)
    output_box.config(state="normal")  # enable editing
    output_box.insert(tk.END, f"> {user_input}\n{response}\n\n")
    output_box.config(state="disabled")  # lock editing
    entry.delete(0, tk.END)

# main window
root = tk.Tk()
root.title("Pokédex")
root.configure(bg="#d62828")
root.iconbitmap("rotom.ico")

# input 
entry = tk.Entry(root, width=50, font=("Consolas", 12), relief="solid", bd=3)
entry.grid(row=0, column=0, padx=10, pady=10)

# ask button
ask_button = tk.Button(root, text="Ask Pokédex", command=ask_question, bg="#cfc539", fg="white", font=("Arial", 12, "bold"), relief="raised", bd=4)
ask_button.grid(row=0, column=1, padx=5, pady=10)

# output box
output_box = scrolledtext.ScrolledText(root, wrap=tk.WORD, width=60, height=20, font=("Consolas", 11))
output_box.grid(row=1, column=0, columnspan=2, padx=10, pady=10)
output_box.config(state="disabled")

root.mainloop()

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Procedure.ProcedureWarning} {category: GENERIC} {title: The query used a procedure that generated a warning.} {description: The query used a procedure that generated a warning. (The field `propertyTypes` will change output format in the next major version.)} {position: line: 2, column: 13, offset: 13} for query: '\n            CALL db.schema.nodeTypeProperties()\n            YIELD nodeLabels, propertyName, propertyTypes\n            RETURN nodeLabels, propertyName, propertyTypes\n        '
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Procedure.ProcedureWarning} {category: GENERIC} {title: The query used a procedure that generated a warning.} {description: The query used a procedure that generated a warning. (The field `propertyTypes` will change output format in the next major version.)} {position: line: 2, column: 13, offset: 13} for query: '\n        

Generated Cypher Query:
 MATCH (pidgey:Pokemon {name: 'Pidgey'})-[:HAS_MOVE]->(move:Move)-[:HAS_TYPE]->(moveType:Type),
      (bulbasaur:Pokemon {name: 'Bulbasaur'})-[:HAS_TYPE]->(bulbasaurType:Type),
      (moveType)-[effectiveness:EFFECTIVENESS]->(bulbasaurType)
WHERE effectiveness.factor = 2
RETURN move.name
Query Results:
 [{'move.name': 'Gust'}, {'move.name': 'Wing-attack'}, {'move.name': 'Fly'}, {'move.name': 'Sand-attack'}, {'move.name': 'Toxic'}, {'move.name': 'Agility'}, {'move.name': 'Reflect'}, {'move.name': 'Mirror-move'}, {'move.name': 'Sky-attack'}, {'move.name': 'Rest'}, {'move.name': 'Mud-slap'}, {'move.name': 'Sunny-day'}, {'move.name': 'Heat-wave'}, {'move.name': 'Feather-dance'}, {'move.name': 'Air-cutter'}, {'move.name': 'Aerial-ace'}, {'move.name': 'Roost'}, {'move.name': 'Pluck'}, {'move.name': 'Tailwind'}, {'move.name': 'U-turn'}, {'move.name': 'Air-slash'}, {'move.name': 'Brave-bird'}, {'move.name': 'Defog'}, {'move.name': 'Hurricane'}]


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Procedure.ProcedureWarning} {category: GENERIC} {title: The query used a procedure that generated a warning.} {description: The query used a procedure that generated a warning. (The field `propertyTypes` will change output format in the next major version.)} {position: line: 2, column: 13, offset: 13} for query: '\n            CALL db.schema.nodeTypeProperties()\n            YIELD nodeLabels, propertyName, propertyTypes\n            RETURN nodeLabels, propertyName, propertyTypes\n        '
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Procedure.ProcedureWarning} {category: GENERIC} {title: The query used a procedure that generated a warning.} {description: The query used a procedure that generated a warning. (The field `propertyTypes` will change output format in the next major version.)} {position: line: 2, column: 13, offset: 13} for query: '\n        

Generated Cypher Query:
 MATCH (p1:Pokemon {name: 'Pidgey'})-[:HAS_MOVE]->(m:Move)-[:HAS_TYPE]->(t:Type)<-[:HAS_TYPE]-(p2:Pokemon {name: 'Bulbasaur'}),
      (t)-[e:EFFECTIVENESS]->(t2:Type)<-[:HAS_TYPE]-(p2)
WHERE e.factor > 1
RETURN m.name, m.power, m.accuracy, e.factor
ORDER BY e.factor DESC, m.power DESC, m.accuracy DESC
Query Results:
 [{'m.name': 'Toxic', 'm.power': nan, 'm.accuracy': 90.0, 'e.factor': 2}]
